# Linear Regression: From First Principles

**Learning goal:** explain ordinary least squares (OLS), implement it with NumPy, compare it with scikit-learn, check correctness, and reason about scaling. This notebook focuses on linear regression; ridge, lasso, elastic net, logistic regression, GLMs, LDA, and QDA should each receive a dedicated treatment rather than being squeezed into one example.

**Suggested workflow:** run each cell in order, change a parameter, predict what will happen, then run the cell and compare.

## 1. Definition and intuition

Linear regression predicts a continuous target using a weighted sum of features. With one input feature it fits a line; with multiple features it fits a hyperplane. It chooses coefficients that minimize the sum of squared residuals.

A useful mental model: among all linear predictions available from the chosen features, OLS selects the one whose prediction vector is closest to the observed target vector in Euclidean distance.

## 2. Mathematics

Let $X\in\mathbb{R}^{n\times d}$ be a feature matrix and $y\in\mathbb{R}^n$ the target. With coefficients $w\in\mathbb{R}^d$ and intercept $b$, predictions are

$$\hat y = Xw + b\mathbf{1}.$$

The OLS objective is

$$J(w,b)=\frac{1}{n}\sum_{i=1}^{n}(x_i^T w+b-y_i)^2.$$

Append a ones column to form $A=[\mathbf{1}\;X]$ and combine the parameters into $\theta=[b,w^T]^T$. Then

$$J(\theta)=\frac1n\|A\theta-y\|_2^2,\qquad \nabla J(\theta)=\frac{2}{n}A^T(A\theta-y).$$

At a stationary point, $A^T A\theta=A^Ty$. This is the normal equation. In code, prefer `np.linalg.lstsq(A, y)` to explicitly computing $(A^TA)^{-1}A^Ty$: explicit inversion is less numerically stable and is unnecessary.

## 3. Why it works and its assumptions

The squared-error objective is a convex quadratic. Therefore any global minimizer is a solution to the normal equations. If the design matrix has full column rank, the minimizer is unique. If columns are linearly dependent, coefficients may not be unique even though fitted values can be.

For statistical interpretation, a common assumption is that the conditional mean is linear in the selected features and the error has zero conditional mean. Gaussian errors are not required just to compute OLS; they support the familiar Gaussian-noise likelihood interpretation and some exact finite-sample inference results.

**Failure modes:** outliers exert large influence because errors are squared; nonlinear relationships can be missed; multicollinearity makes coefficient estimates unstable; and extrapolation beyond observed data can be unreliable.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

SEED = 42
rng = np.random.default_rng(SEED)
print("NumPy:", np.__version__)

## 4. Create a toy dataset

We generate data from $y=3+2x+\epsilon$, where $\epsilon$ is random noise. Since the observations are noisy, the estimates will not equal the generating values exactly.

In [ ]:
X = rng.uniform(-3, 3, size=(200, 1))
y = 3 + 2 * X[:, 0] + rng.normal(0, 1, size=200)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED
)

plt.scatter(X_train[:, 0], y_train, s=18, alpha=0.7, label="train")
plt.scatter(X_test[:, 0], y_test, s=18, alpha=0.7, label="test")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.title("Synthetic linear relationship with noise")
plt.show()

## 5. Implement OLS from scratch with NumPy

The function below validates inputs, adds an intercept column, and calls least-squares decomposition. It returns coefficients and diagnostic rank information. `lstsq` is a numerical linear algebra routine—not an ML estimator—so the modelling logic remains explicit.

In [ ]:
def fit_ols(X, y):
    """Fit ordinary least squares. X must be 2D; y must be 1D."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    if X.ndim != 2:
        raise ValueError("X must be a 2D array")
    if y.ndim != 1 or len(y) != len(X):
        raise ValueError("y must be 1D and match X's row count")
    if len(X) == 0 or X.shape[1] == 0:
        raise ValueError("X must have at least one sample and one feature")
    if not np.isfinite(X).all() or not np.isfinite(y).all():
        raise ValueError("X and y must contain only finite values")

    design = np.column_stack([np.ones(len(X)), X])
    theta, residuals, rank, singular_values = np.linalg.lstsq(design, y, rcond=None)
    diagnostics = {"rank": int(rank), "singular_values": singular_values}
    return float(theta[0]), theta[1:].copy(), diagnostics


def predict_ols(X, intercept, coef):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or X.shape[1] != len(coef):
        raise ValueError("X feature count must match coefficient count")
    return X @ coef + intercept

intercept, coef, diagnostics = fit_ols(X_train, y_train)
y_pred = predict_ols(X_test, intercept, coef)
print(f"intercept estimate: {intercept:.3f}")
print(f"coefficient estimate: {coef}")
print(f"test MSE: {mean_squared_error(y_test, y_pred):.3f}")
print(f"test R²: {r2_score(y_test, y_pred):.3f}")
print("design-matrix rank:", diagnostics["rank"])

## 6. Compare with scikit-learn

Use exactly the same data split. The fitted predictions should agree within floating-point tolerance because both methods solve the same least-squares problem. Coefficients can differ slightly on poorly conditioned problems due to solver details.

In [ ]:
sk_model = LinearRegression().fit(X_train, y_train)
sk_pred = sk_model.predict(X_test)
print("scikit-learn intercept:", sk_model.intercept_)
print("scikit-learn coefficients:", sk_model.coef_)
print("maximum absolute prediction difference:", np.max(np.abs(y_pred - sk_pred)))
assert np.allclose(y_pred, sk_pred, atol=1e-9)

## 7. Gradient descent: a second way to solve the same problem

OLS can also be fit iteratively by following the negative gradient. This is useful conceptually because many ML methods are optimized iteratively. The learning rate controls step size: too large may diverge, while too small converges slowly. Feature scaling often improves conditioning when features are on different scales.

In [ ]:
def fit_ols_gradient_descent(X, y, learning_rate=0.05, epochs=3000):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    if X.ndim != 2 or y.ndim != 1 or len(X) != len(y):
        raise ValueError("Expected X shape (n, d) and y shape (n,)")
    if learning_rate <= 0 or epochs < 1:
        raise ValueError("learning_rate and epochs must be positive")
    design = np.column_stack([np.ones(len(X)), X])
    theta = np.zeros(design.shape[1])
    losses = []
    for _ in range(epochs):
        error = design @ theta - y
        gradient = (2 / len(y)) * (design.T @ error)
        theta -= learning_rate * gradient
        losses.append(float(np.mean((design @ theta - y) ** 2)))
    return float(theta[0]), theta[1:].copy(), losses

gd_intercept, gd_coef, losses = fit_ols_gradient_descent(X_train, y_train)
print("gradient-descent intercept:", round(gd_intercept, 3))
print("gradient-descent coefficient:", np.round(gd_coef, 3))
plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("training MSE")
plt.title("Gradient descent convergence")
plt.show()

## 8. Polynomial regression: nonlinear features, linear parameters

Polynomial regression adds features such as $x^2$ and $x^3$, then uses a linear estimator on that expanded representation. The prediction is nonlinear in the original input, but linear in the coefficients. Increasing polynomial degree can reduce training error while worsening test error.

In [ ]:
Xp = rng.uniform(-2, 2, size=(300, 1))
yp = 1 - 0.8 * Xp[:, 0] + 1.4 * Xp[:, 0]**2 + rng.normal(0, 0.6, 300)
Xtr, Xte, ytr, yte = train_test_split(Xp, yp, test_size=0.25, random_state=SEED)
for degree in (1, 2, 5, 12):
    model = make_pipeline(PolynomialFeatures(degree, include_bias=False), LinearRegression())
    model.fit(Xtr, ytr)
    print(f"degree={degree:2d} | train MSE={mean_squared_error(ytr, model.predict(Xtr)):.3f} | test MSE={mean_squared_error(yte, model.predict(Xte)):.3f}")

## 9. Best practices and hyperparameters

- OLS itself has no usual regularization-strength hyperparameter. Feature selection and feature expansion control model flexibility.
- For polynomial regression, tune `degree` using validation data, not the test set.
- Use a `Pipeline` to fit transformations only on training folds and avoid leakage.
- Scaling is not necessary for the OLS least-squares definition, but often helps gradient-based solvers and is important when comparing regularized coefficients.
- Diagnose residuals, outliers, heteroscedasticity, multicollinearity, and performance on held-out data.
- Consider Ridge, Lasso, or Elastic Net when regularization is needed; their objectives and coefficient behavior deserve separate implementations.

## 10. Complexity: training, inference, and memory

Let $n$ be the number of rows and $d$ the number of features.

| Method | Training time (dense, typical) | Model/persistent space | Batch prediction time |
|---|---:|---:|---:|
| Normal equations via explicit Gram matrix | $O(nd^2+d^3)$ | $O(d^2)$ Gram matrix plus input | $O(md)$ |
| QR/SVD least squares (`lstsq`, typical when $n\ge d$) | roughly $O(nd^2)$ | input $O(nd)$, workspaces vary | $O(md)$ |
| Gradient descent for $T$ full-batch steps | $O(Tnd)$ | $O(nd+d)$ for dense data | $O(md)$ |

The fitted model stores $O(d)$ coefficients. These are asymptotic estimates; solver choice, sparsity, conditioning, and whether $d$ exceeds $n$ change the practical cost. Never explicitly invert a matrix merely to apply the normal equation.

## 11. Scaling to millions of samples

For millions of rows, start with a benchmark of a trusted solver, track memory as well as runtime, and exploit sparse input when features are sparse. Full-batch gradient descent costs $O(nd)$ per epoch; stochastic/minibatch methods can update from subsets of observations and may be suitable when exact closed-form solving is expensive. Avoid creating unnecessary copies of a large design matrix. For inference, predict in row chunks if loading all rows at once is a memory problem.

In [ ]:
def benchmark_ols(n_samples, n_features=10, seed=0):
    local_rng = np.random.default_rng(seed)
    Xb = local_rng.normal(size=(n_samples, n_features))
    true_w = local_rng.normal(size=n_features)
    yb = Xb @ true_w + local_rng.normal(scale=0.1, size=n_samples)
    model = LinearRegression()
    start = time.perf_counter()
    model.fit(Xb, yb)
    seconds = time.perf_counter() - start
    return {
        "n_samples": n_samples,
        "n_features": n_features,
        "fit_seconds": round(seconds, 4),
        "training_mse": round(mean_squared_error(yb, model.predict(Xb)), 6),
    }

# Start modestly; increase sizes only if your machine has enough memory.
[benchmark_ols(n) for n in (1_000, 10_000, 100_000)]

## 12. GPU: when it may help

Dense linear algebra can benefit from a GPU for sufficiently large workloads, but data transfer and solver availability matter. GPU is not automatically faster for small tabular data. The next cell gives an optional PyTorch full-batch gradient-descent version. It uses CUDA if available and otherwise runs on CPU; this is an educational optimizer, not a claim that it outperforms scikit-learn.

In [ ]:
try:
    import torch

    def fit_ols_torch(X, y, learning_rate=0.05, epochs=1000):
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        X_t = torch.as_tensor(np.asarray(X), dtype=torch.float32, device=device)
        y_t = torch.as_tensor(np.asarray(y), dtype=torch.float32, device=device).reshape(-1)
        design = torch.cat([torch.ones((len(X_t), 1), device=device), X_t], dim=1)
        theta = torch.zeros(design.shape[1], device=device, requires_grad=True)
        optimizer = torch.optim.SGD([theta], lr=learning_rate)
        for _ in range(epochs):
            optimizer.zero_grad()
            loss = torch.mean((design @ theta - y_t) ** 2)
            loss.backward()
            optimizer.step()
        fitted = theta.detach().cpu().numpy()
        return float(fitted[0]), fitted[1:], str(device)

    torch_intercept, torch_coef, torch_device = fit_ols_torch(X_train, y_train)
    print("PyTorch device:", torch_device)
    print("PyTorch estimates:", torch_intercept, torch_coef)
except ImportError:
    print("PyTorch is not installed; skip the optional GPU section.")

## 13. Correctness checks and synthesis

The core idea: linear regression projects the target vector onto the column space of the design matrix under squared-error loss. At a least-squares optimum, the residual vector is orthogonal to every column of the design matrix. The normal-equation solver and gradient descent are different ways of minimizing the same convex objective.

In [ ]:
# Reproducibility and invalid-input checks
b1, w1, _ = fit_ols(X_train, y_train)
b2, w2, _ = fit_ols(X_train, y_train)
assert b1 == b2 and np.array_equal(w1, w2)
assert np.isfinite(y_pred).all()
assert mean_squared_error(y_test, y_pred) < np.var(y_test)
try:
    fit_ols(np.array([1, 2, 3]), np.array([1, 2, 3]))
except ValueError:
    pass
else:
    raise AssertionError("Expected a ValueError for one-dimensional X")
print("All OLS checks passed.")